In [4]:
import os
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(
    model="openai/gpt-4o-mini",
    temperature=0
)

response = llm.invoke("Hi")
print(response.content)

Hello! How can I assist you today?


## Utils

In [7]:
## DocLoader
from langchain_community.document_loaders import PyMuPDFLoader
def resume_loader(file_path):
    loader = PyMuPDFLoader(file_path)
    doc_list =  loader.load()
    return '\n'.join([i.page_content for i in doc_list if i.page_content])

def text_loader(file_path):
    with open(file_path, 'r') as f:
        return f.read()

## Analyzer

In [27]:
### Resume Analyzer
resume_text = resume_loader("/Users/munna/Projects/QBS/webiner_project/Mahmud_Hasan_Munna_BL.pdf")
jd_text = text_loader("/Users/munna/Projects/QBS/webiner_project/jd.txt")


In [40]:
### Promt --> LLM --> Structured Output Response
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage,SystemMessage
from pydantic import BaseModel, Field
from typing import List

In [63]:
def analyze_resume(resume_text, jd_text):
    class ResumeAnalysis(BaseModel):
        strong_points: List[str] = Field(description="List of strong points in the resume")
        weak_points: List[str] = Field(description="List of weak points in the resume")

    analyze_llm = llm.with_structured_output(ResumeAnalysis)
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
                You are an expert AI assistant specializing in Resume Strong and Weakness Analysis.

                Responsibilities:
                - You will be provided with a resume and a job description.
                - Analyze the resume in the context of the job description.
                - You will provide a structured analysis highlighting the strong points and weak points of the resume with respect to the job description.
                - Max allowed strong points: 5
                - Max allowed weak points: 5

                """
            ),
            (
                "human",
                """
                Resume:
                {resume}

                JD:
                {jd}
                """
            ),
        ]
    )
    analyze_chain = prompt | analyze_llm
    result = analyze_chain.invoke(
        {
            "resume": resume_text,
            "jd": jd_text,
        }
    )
    return dict(result)

In [64]:
analyze_resume(resume_text, jd_text)

{'strong_points': ['4+ years of experience in AI/ML engineering with a focus on deploying production-grade machine learning systems.',
  "Hands-on expertise in LLM/GenAI systems, including RAG and agent orchestration, which aligns with the job's requirements.",
  'Experience with cloud platforms (AWS) and familiarity with cloud-native ML deployment, which is essential for the role.',
  'Proven track record in automating processes and improving efficiency, demonstrated by the reduction in onboarding time and manual work hours.',
  'Strong technical skills in Python, FastAPI, Docker, and CI/CD workflows, which are crucial for architecting scalable AI solutions.'],
 'weak_points': ['Only 4 years of experience, while the job requires 7+ years in AI/ML engineering and enterprise AI platforms.',
  'Limited experience in leading technical teams or AI engineering squads, which is a key requirement for the role.',
  'No mention of experience with GPU-based inference systems or fine-tuning techn